In [2]:
#library import

import numpy as np
import pandas as pd
import os as os
from os.path import exists
from astropy.utils.data import download_file
from astropy.io import fits
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
import json
from astropy.table import Table
import requests
import sys
from sseclient import SSEClient
import astropy
import os
import cv2
from PIL import Image,UnidentifiedImageError
from io import BytesIO
import sys

In [3]:
#params control

params = {
    "target": "65P",
    "sources": "neat_palomar_tricam",
    "cached": "true",
    "format": "json"
}

sources_list = ["neat_palomar_tricam","neat_maui_geodss","skymapper_dr4","ps1_dr2","catalina_bigelow",
                "catalina_lemmon","catalina_bok_neo_survey","spacewatch","loneos","atlas_haleakela",
                "atlas_mauna_loa","atlas_rio_hurtado","atlas_surtherland"]
#we can temporary ignore PanSTARRS (ps1_dr2)

potential_sp_target_list = ["1P","2P","45P","65P","67P","96P"]
new_target_list = ["9P","17P","19P","29P","41P","46P","73P","81P","103P","133P","238P","288P","311P","313P","324P","354P"]
C_target_list = ["C/1995 O1","C/2017 E4", "C/2017 K2", "C/2019 Y4", "C/2020 F3", "C/2021 A1", "C/2022 E3", "C/2023 A3", "P/2013 R3"]

In [4]:
#api request funtion (which is used to get data with specific target, source updated with params)

def api_request(params):
    base_url = "https://catch-api.astro.umd.edu"
    res = requests.get(base_url + "/catch", params=params)
    data = res.json()
    if res.status_code !=200:
        return None
    else:
        if data["queued"]: #what is queued?
            messages = SSEClient(base_url + "/stream")
            for message in messages:
                # ignore blank lines
                if message.data == "":
                    continue

                message_data = json.loads(message.data)

                # edit out keep-alive messages
                if not isinstance(message_data, dict):
                    continue

                if message_data["job_prefix"] == data["job_id"][:8]:
                    # this message is for us, print the text
                    print(message_data['text'], file=sys.stderr)
                else:
                    continue

                # Message status may be "success", "error", "running", "queued".
                if message_data["status"] in ["error", "success"]:
                    break
    #"results" is the URL to the search results
        res = requests.get(data["results"])
        data = res.json()
        return data #data is JSOn formatted 
        #data is a json file included every value of the object 
        #(archive url (fits file), cutout url (fits file), preview url (which is jpeg))
        #we should care about source, source_name, date, dec, ra


        #sample output (aka data)

        #{'count': 5, 'data': [{'airmass': 1.038482, 'archive_url': 'https://sbnarchive.psi.edu/pds4/surveys/gbo.ast.neat.survey/data_tricam/p20020222/obsdata/20020222120052c.fit.fz', 
        #'cutout_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222120052c?ra=174.62244&dec=17.97594&size=7.20arcmin&format=fits', 
        # 'date': '2002-02-22 12:01:22.000', 'ddec': 13.76413, 'dec': 17.97594, 'delta': 2.49069892940669, 
        # 'diff_url': None, 'dra': -23.1946, 'drh': -5.1794841, 'elong': 159.9563, 'exposure': 60.0, 'filter': 'NONE', 
        # 'fov': '175.563276:18.428626,175.559407:17.284060,174.360705:17.284060,174.356834:18.428625', 
        # 'maglimit': None, 'mjd_start': 52327.500601851854, 'mjd_stop': 52327.501296296294, 'phase': 5.6655, 
        # 'preview_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222120052c?ra=174.62244&dec=17.97594&size=7.20arcmin&format=jpeg', 
        # 'product_id': 'urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222120052c', 
        # 'ra': 174.62244, 'rh': 3.436865439347, 'sangle': 69.62299999999999, 'seeing': None, 
        # 'source': 'neat_palomar_tricam', 'source_name': 'NEAT Palomar Tricam', 'true_anomaly': 258.8381, 
        # 'unc_a': 5.65, 'unc_b': 0.393, 'unc_theta': -24.108, 'vangle': 114.76100000000002, 'vmag': 17.001}, {'airmass': 1.038667, 'archive_url': 'https://sbnarchive.psi.edu/pds4/surveys/gbo.ast.neat.survey/data_tricam/p20020222/obsdata/20020222121552c.fit.fz', 'cutout_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222121552c?ra=174.62074&dec=17.9769&size=7.20arcmin&format=fits', 'date': '2002-02-22 12:16:22.000', 'ddec': 13.75189, 'dec': 17.9769, 'delta': 2.49063084002561, 'diff_url': None, 'dra': -23.1647, 'drh': -5.179512, 'elong': 159.9648, 'exposure': 60.0, 'filter': 'NONE', 'fov': '175.561866:18.426662,175.557998:17.282097,174.359308:17.282096,174.355438:18.426661', 'maglimit': None, 'mjd_start': 52327.51101851852, 'mjd_stop': 52327.511712962965, 'phase': 5.6632, 'preview_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222121552c?ra=174.62074&dec=17.9769&size=7.20arcmin&format=jpeg', 'product_id': 'urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222121552c', 'ra': 174.62074, 'rh': 3.436834277647, 'sangle': 69.596, 'seeing': None, 'source': 'neat_palomar_tricam', 'source_name': 'NEAT Palomar Tricam', 'true_anomaly': 258.8396, 'unc_a': 5.65, 'unc_b': 0.393, 'unc_theta': -24.107, 'vangle': 114.75999999999999, 'vmag': 17.001}, {'airmass': 1.055449, 'archive_url': 'https://sbnarchive.psi.edu/pds4/surveys/gbo.ast.neat.survey/data_tricam/p20020121/obsdata/20020121132624c.fit.fz', 'cutout_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020121_obsdata_20020121132624c?ra=177.51011&dec=15.25013&size=7.20arcmin&format=fits', 'date': '2002-01-21 13:26:54.000', 'ddec': 9.813682, 'dec': 15.25013, 'delta': 2.83330835056674, 'diff_url': None, 'dra': -2.64437, 'drh': -5.0789549, 'elong': 128.5424, 'exposure': 60.0, 'filter': 'NONE', 'fov': '178.359720:15.349569,178.356602:14.205001,177.175929:14.205000,177.172809:15.349568', 'maglimit': None, 'mjd_start': 52295.56, 'mjd_stop': 52295.560694444444, 'phase': 12.5942, 'preview_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020121_obsdata_20020121132624c?ra=177.51011&dec=15.25013&size=7.20arcmin&format=jpeg', 'product_id': 'urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020121_obsdata_20020121132624c', 'ra': 177.51011, 'rh': 3.531535016403, 'sangle': 103.483, 'seeing': None, 'source': 'neat_palomar_tricam', 'source_name': 'NEAT Palomar Tricam', 'true_anomaly': 254.1847, 'unc_a': 4.967, 'unc_b': 0.359, 'unc_theta': -25.651, 'vangle': 116.10500000000002, 'vmag': 17.36}, {'airmass': 1.055584, 'archive_url': 'https://sbnarchive.psi.edu/pds4/surveys/gbo.ast.neat.survey/data_tricam/p20020121/obsdata/20020121134124c.fit.fz', 'cutout_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020121_obsdata_20020121134124c?ra=177.50992&dec=15.25081&size=7.20arcmin&format=fits', 'date': '2002-01-21 13:41:54.000', 'ddec': 9.806215, 'dec': 15.25081, 'delta': 2.83316207193636, 'diff_url': None, 'dra': -2.62949, 'drh': -5.0789922, 'elong': 128.5529, 'exposure': 60.0, 'filter': 'NONE', 'fov': '178.367220:15.349569,178.364102:14.205001,177.183429:14.205000,177.180309:15.349568', 'maglimit': None, 'mjd_start': 52295.57041666667, 'mjd_stop': 52295.57111111111, 'phase': 12.5924, 'preview_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020121_obsdata_20020121134124c?ra=177.50992&dec=15.25081&size=7.20arcmin&format=jpeg', 'product_id': 'urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020121_obsdata_20020121134124c', 'ra': 177.50992, 'rh': 3.531504458169, 'sangle': 103.47899999999998, 'seeing': None, 'source': 'neat_palomar_tricam', 'source_name': 'NEAT Palomar Tricam', 'true_anomaly': 254.1862, 'unc_a': 4.967, 'unc_b': 0.359, 'unc_theta': -25.65, 'vangle': 116.10500000000002, 'vmag': 17.36}, {'airmass': 1.038863, 'archive_url': 'https://sbnarchive.psi.edu/pds4/surveys/gbo.ast.neat.survey/data_tricam/p20020222/obsdata/20020222123100c.fit.fz', 'cutout_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222123100c?ra=174.61904&dec=17.97786&size=7.20arcmin&format=fits', 'date': '2002-02-22 12:31:30.000', 'ddec': 13.74028, 'dec': 17.97786, 'delta': 2.49056228151862, 'diff_url': None, 'dra': -23.1321, 'drh': -5.1795403, 'elong': 159.9733, 'exposure': 60.0, 'filter': 'NONE', 'fov': '175.553447:18.426101,175.549579:17.281536,174.350893:17.281535,174.347023:18.426100', 'maglimit': None, 'mjd_start': 52327.521527777775, 'mjd_stop': 52327.52222222222, 'phase': 5.6609, 'preview_url': 'https://sbnsurveys.astro.umd.edu/api/images/urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222123100c?ra=174.61904&dec=17.97786&size=7.20arcmin&format=jpeg', 'product_id': 'urn:nasa:pds:gbo.ast.neat.survey:data_tricam:p20020222_obsdata_20020222123100c', 'ra': 174.61904, 'rh': 3.436802838785, 'sangle': 69.56899999999999, 'seeing': None, 'source': 'neat_palomar_tricam', 'source_name': 'NEAT Palomar Tricam', 'true_anomaly': 258.8412, 'unc_a': 5.65, 'unc_b': 0.393, 'unc_theta': -24.107, 'vangle': 114.75999999999999, 'vmag': 17.001}], 'job_id': 'ceecb91939ed42aaa1e00cb824e1a586', 'parameters': {'padding': 0.0, 'start_date': None, 'stop_date': None, 'target': '65P', 'uncertainty_ellipse': False}, 'status': [{'count': 5, 'date': '2026-07-06 21:56:48.713', 'execution_time': None, 'source': 'neat_palomar_tricam', 'source_name': 'NEAT Palomar Tricam', 'status': 'finished'}], 'version': '3.0.0'}

        



In [ ]:
#some reference code


# cutout_urls = [entry["cutout_url"] for entry in data["data"]]
# #sample of how to extract a value category from data

# preview_url = [entry["preview_url"] for entry in data["data"]]

# ## CATCH API
# # image_url = cutout_urls[0]
# # image_file = download_file(image_url, cache=True)
# # hdulist = fits.open(image_file)
# # image = astropy.open(image_file) 

#dictionary method references
# .get(key)     return value from that key
# .keys()       list out all keys
# .values()     list out all values
# .items()      return dict as tuples
# .clear()      remove all items
# ["key"] = ?   update value of the specific key

In [5]:
def download_file(url, local_filename):
    with requests.get(url, stream=True) as response: #response = url, this line will get the url ready
        if response.status_code ==200:
            with open(local_filename, 'wb') as file:
                file.write(response.content)   
        else:
            print("download failed")    

# Example usage:
# download_file(preview_url[0], save_as)
# print("Web file downloaded successfully!")


In [4]:
#create folder
def create_folder(folder_dir):
    if os.path.isdir(folder_dir):
        sys.exit("Error: The folder exit, please remove the old folder")
    else:
        os.makedirs(folder_dir)    

In [48]:
#reset dir to training data set
os.chdir(r"\\wsl.localhost\Ubuntu\home\tuannbas\GRADMAP_SS26\GRADMAP_SS26\training_dataset_folder")
main_path = os.getcwd()

In [49]:
print(os.getcwd())

\\wsl.localhost\Ubuntu\home\tuannbas\GRADMAP_SS26\GRADMAP_SS26\training_dataset_folder


In [ ]:
# params = {
#     "target": "1P",
#     "sources": "neat_palomar_tricam",
#     "cached": "true",
#     "format": "json"
# }
# preview_urls = [entry["preview_url"] for entry in api_request(params)["data"]]
# response = requests.get(preview_urls[0])
# image = Image.open(BytesIO(response.content))
# width, height = image.size 
# center_width = width//2
# center_height = height//2
# crop_width = 50
# crop_height = 50
# left = center_width-crop_width
# right = center_width+crop_width
# bottom = center_height+crop_height
# top = center_height-crop_height
# cropped = image.crop((left, top, right, bottom))
# cropped.save("cropped_pillow.jpg")



#comet_images(params,potential_sp_target_list[1],sources_list[0],os.getcwd())

In [68]:
#procedure to process CATCH images (crop)


def crop_images(params,comet_name,sources,main_path,new_folder_name = 0):
    #use api to get image (jpeg), then crop them with cv2, then save them into computer
    params["target"] = comet_name
    if new_folder_name == 0:
        main_folder_name = comet_name + "_" + "images"
        create_folder(main_folder_name)
        main_folder_path = main_path + "\\" + str(main_folder_name)
        os.chdir(main_folder_path)
    else:
        create_folder(new_folder_name)
        main_folder_path = main_path + "\\" + str(new_folder_name)
        os.chdir(main_folder_path)
    for source in sources:
        params["sources"] = source
        #api request here
        if "/" in comet_name:
            index = comet_name.find("/")
            comet_name = comet_name[:index]+"_"+comet_name[index+1:]
        data_request = api_request(params)
        if data_request != None:
            preview_urls = [entry["preview_url"] for entry in data_request["data"]]
            i=0
            for preview_url in preview_urls:
                try:
                    comet_file_name = comet_name + "_" + params["sources"] + "_" + str(i)
                    response = requests.get(preview_url,timeout=(10,60))
                    image = Image.open(BytesIO(response.content))
                    width, height = image.size 
                    center_width = width//2
                    center_height = height//2
                    crop_width = 50
                    crop_height = 50
                    cropped = image.crop((center_width-crop_width, center_height-crop_height, center_width+crop_width, center_height+crop_height))
                    cropped.save(comet_file_name+".jpg")
                    i+=1
                except UnidentifiedImageError:
                    continue
        os.chdir(main_folder_path)

                
    os.chdir(main_path)
    print("Finish downloading cropped images of " + comet_name)



In [69]:
params = {
    "target": "65P",
    "sources": "neat_palomar_tricam",
    "cached": "true",
    "format": "json"
}

crop_images(params,potential_sp_target_list[3],sources_list,main_path)

KeyboardInterrupt: 

In [10]:
crop_images(params,potential_sp_target_list[0],sources_list,main_path)

Finish downloading cropped images of 1P


In [16]:
crop_images(params,potential_sp_target_list[1],sources_list,main_path)

Finish downloading cropped images of 2P


In [17]:
crop_images(params,potential_sp_target_list[2],sources_list,main_path)

Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 68.3 deg over 478.0 days
NEAT Palomar Tricam: Caught 3 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 130.1 deg over 771.0 days
NEAT Maui GEODSS: Caught 9 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 607.1 deg over 2745.0 days
SkyMapperDR4: Caught 59 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Bigelow: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Bigelow: Searching 554.6 deg over 2215.0 days
Catalina Sky Survey, Mt. Bigelow: Caught 324 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 554.6 deg over 2215.0 days
Catal

Finish downloading cropped images of 45P


In [18]:
crop_images(params,potential_sp_target_list[4],sources_list,main_path)

Finish downloading cropped images of 67P


In [19]:
crop_images(params,potential_sp_target_list[5],sources_list,main_path)

Finish downloading cropped images of 96P


In [26]:
new_target_list = ["9P","17P","19P","29P","41P","46P","73P","81P","103P","133P","238P","288P","311P","313P","324P","354P"]

In [40]:

crop_images(params,new_target_list[0],sources_list,main_path)


Finish downloading cropped images of 9P


In [41]:
for item in new_target_list:
    if item not in ["9P","17P","19P","29P"]:
        crop_images(params,item,sources_list,main_path)


Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 81.9 deg over 478.0 days
NEAT Palomar Tricam: Caught 6 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 115.7 deg over 771.0 days
NEAT Maui GEODSS: Caught 6 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 637.7 deg over 2745.0 days
SkyMapperDR4: Caught 44 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Bigelow: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Bigelow: Searching 548.4 deg over 2215.0 days
Catalina Sky Survey, Mt. Bigelow: Caught 824 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 548.4 deg over 2215.0 days
Catal

Finish downloading cropped images of 41P


Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 284.6 deg over 478.0 days
NEAT Palomar Tricam: Caught 0 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 334.9 deg over 771.0 days
NEAT Maui GEODSS: Caught 42 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 636.7 deg over 2745.0 days
SkyMapperDR4: Caught 66 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Bigelow: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Bigelow: Searching 553.2 deg over 2215.0 days
Catalina Sky Survey, Mt. Bigelow: Caught 268 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 553.2 deg over 2215.0 days
Cat

Finish downloading cropped images of 46P
Finish downloading cropped images of 73P
Finish downloading cropped images of 81P
Finish downloading cropped images of 103P
Finish downloading cropped images of 133P
Finish downloading cropped images of 238P
Finish downloading cropped images of 288P
Finish downloading cropped images of 311P
Finish downloading cropped images of 313P


Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 100.3 deg over 478.0 days
NEAT Palomar Tricam: Caught 6 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 180.9 deg over 771.0 days
NEAT Maui GEODSS: Caught 0 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 724.4 deg over 2745.0 days
SkyMapperDR4: Caught 32 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Bigelow: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Bigelow: Searching 570.1 deg over 2215.0 days
Catalina Sky Survey, Mt. Bigelow: Caught 384 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 570.1 deg over 2215.0 days
Cata

Finish downloading cropped images of 324P


Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 174.7 deg over 478.0 days
NEAT Palomar Tricam: Caught 0 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 284.5 deg over 771.0 days
NEAT Maui GEODSS: Caught 0 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 922.5 deg over 2745.0 days
SkyMapperDR4: Caught 46 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Bigelow: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Bigelow: Searching 774.8 deg over 2215.0 days
Catalina Sky Survey, Mt. Bigelow: Caught 553 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 774.8 deg over 2215.0 days
Cata

JSONDecodeError: Unterminated string starting at: line 1 column 81 (char 80)

In [58]:
new_target_list = ["9P","17P","19P","29P","41P","46P","73P","81P","103P","133P","238P","288P","311P","313P","324P","354P"]
C_target_list = ["C/1995 O1","C/2017 E4", "C/2017 K2", "C/2019 Y4", "C/2020 F3", "C/2021 A1", "C/2022 E3", "C/2023 A3", "P/2013 R3"]
new_name_list = ["C_1995_O1","C_2017_E4", "C_2017_K2", "C_2019_Y4", "C_2020_F3", "C_2021_A1", "C_2022 E3", "C_2023_A3", "P_2013_R3"]

In [12]:
crop_images(params,"354P",sources_list,main_path)

Finish downloading cropped images of 354P


In [70]:
os.chdir(r"\\wsl.localhost\Ubuntu\home\tuannbas\GRADMAP_SS26\GRADMAP_SS26\training_dataset_folder")
main_path = os.getcwd()

In [62]:
crop_images(params,C_target_list[2],sources_list,main_path,new_name_list[2])

Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 447.3 deg over 2215.0 days
Catalina Sky Survey, Mt. Lemmon: Caught 150 observations
Task complete.
Starting moving target query.
Spacewatch: Query from 2003-03-23 to 2016-12-24.
Spacewatch: Searching 181.5 deg over 5027.0 days
Spacewatch: Caught 0 observations
Task complete.
Starting moving target query.
LONEOS: Query from 2003-08-29 to 2008-03-01.
LONEOS: Searching 45.7 deg over 1648.0 days
LONEOS: Caught 21 observations
Task complete.
Starting moving target query.
ATLAS Hawaii, Haleakela: Query from 2015-08-06 to 2026-04-19.
ATLAS Hawaii, Haleakela: Searching 559.8 deg over 3911.0 days
ATLAS Hawaii, Haleakela: Caught 1601 observations
Task complete.
Starting moving target query.
ATLAS Hawaii, Mauna Loa: Query from 2017-02-02 to 2026-04-07.
ATLAS Hawaii, Mauna Loa: Searching 529.1 deg over 3353.0 days
ATLAS Hawaii, Mauna Loa: Caught 1576 observ

Finish downloading cropped images of C_2017 K2


In [63]:
crop_images(params,C_target_list[3],sources_list,main_path,new_name_list[3])

Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 8.6 deg over 478.0 days
NEAT Palomar Tricam: Caught 27 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 10.6 deg over 771.0 days
NEAT Maui GEODSS: Caught 3 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 391.4 deg over 2745.0 days
SkyMapperDR4: Caught 6 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Bigelow: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Bigelow: Searching 334.4 deg over 2215.0 days
Catalina Sky Survey, Mt. Bigelow: Caught 112 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 334.4 deg over 2215.0 days
Catalin

Finish downloading cropped images of C_2019 Y4


In [71]:
for i in range(4,len(C_target_list)+1):
    crop_images(params,C_target_list[i],sources_list,main_path,new_name_list[i])

Finish downloading cropped images of C_2020 F3
Finish downloading cropped images of C_2021 A1
Finish downloading cropped images of C_2022 E3
Finish downloading cropped images of C_2023 A3


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [ ]:
crop_images(params,C_target_list[3],sources_list,main_path,new_name_list[3])

In [ ]:
crop_images(params,C_target_list[4],sources_list,main_path,new_name_list[4])

In [ ]:
crop_images(params,C_target_list[5],sources_list,main_path,new_name_list[5])

In [ ]:
crop_images(params,C_target_list[6],sources_list,main_path,new_name_list[6])

In [60]:
for C_target,new_name in zip(C_target_list,new_name_list):
    crop_images(params,C_target,sources_list,main_path,new_name)

Finish downloading cropped images of C_1995 O1


Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 14.0 deg over 478.0 days
NEAT Palomar Tricam: Caught 0 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 18.5 deg over 771.0 days
NEAT Maui GEODSS: Caught 0 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 569.0 deg over 2745.0 days
SkyMapperDR4: Caught 34 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Bigelow: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Bigelow: Searching 127.8 deg over 2215.0 days
Catalina Sky Survey, Mt. Bigelow: Caught 0 observations
Task complete.
Starting moving target query.
Catalina Sky Survey, Mt. Lemmon: Query from 2020-01-01 to 2026-01-22.
Catalina Sky Survey, Mt. Lemmon: Searching 127.8 deg over 2215.0 days
Catalina

Finish downloading cropped images of C_2017 E4


Starting moving target query.
NEAT Palomar Tricam: Query from 2001-11-20 to 2003-03-11.
NEAT Palomar Tricam: Searching 11.6 deg over 478.0 days
NEAT Palomar Tricam: Caught 0 observations
Task complete.
Starting moving target query.
NEAT Maui GEODSS: Query from 1996-04-17 to 1998-05-26.
NEAT Maui GEODSS: Searching 15.9 deg over 771.0 days
NEAT Maui GEODSS: Caught 0 observations
Task complete.
Starting moving target query.
SkyMapperDR4: Query from 2014-03-14 to 2021-09-16.
SkyMapperDR4: Searching 212.1 deg over 2745.0 days
SkyMapperDR4: Caught 0 observations
Task complete.


JSONDecodeError: Unterminated string starting at: line 1 column 36 (char 35)

In [ ]:
#In case you want to remove the folder as they are created before and they can't be removed normally, here
#is a way you can consider to use:
#   If your system is linux, try rm -rf "the path" 
#   If it is windows, try del /f "the path"
#\\wsl.localhost\Ubuntu\home\tuannbas\GRADMAP_SS26\GRADMAP_SS26\training_dataset_folder